In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
shipment = pd.read_csv("cleaned_Shipment.csv")
business = pd.read_csv("cleaned_Business_Metrics.csv")
external = pd.read_csv("cleaned_External_Factors.csv")
status = pd.read_csv("cleaned_Status.csv")
supplier = pd.read_csv("cleaned_Supplier.csv")
transport = pd.read_csv("cleaned_Transportation.csv")
warehouse = pd.read_csv("cleaned_Warehouse.csv")

In [4]:
business.head()

,Shipment_ID,Delivery_Cost,Delay_Hours,Customer_Priority
0,SHP000001,1526.84,1.86,Medium
1,SHP000002,6124.35,17.67,Medium
2,SHP000003,5606.66,5.44,Medium
3,SHP000004,2200.06,14.29,Medium
4,SHP000005,3826.46,1.56,High


In [5]:
transport.head()

,Shipment_ID,Transport_Mode,Route_Distance,Fuel_Cost
0,SHP000001,Rail,328.36,1367.19
1,SHP000002,Rail,1187.29,8815.52
2,SHP000003,Air,1368.36,9264.16
3,SHP000004,Sea,138.63,730.73
4,SHP000005,Road,805.36,4464.22


#### 1.Delay Rate

In [8]:
(business["Delay_Hours"].describe())

count    60000.000000
mean         6.016291
std          4.260936
min          0.010000
25%          2.900000
50%          5.050000
75%          8.070000
max         39.480000
Name: Delay_Hours, dtype: float64

##### How many percentage of shipments were delayed?

##### Formula used - Delay Rate = (Delayed Shipments / Total Shipments) × 100

In [6]:
Delayed_shipments = (business["Delay_Hours"] > 0).sum()
Total_shipments = business["Shipment_ID"].nunique()
Delay_rate = (Delayed_shipments / Total_shipments) * 100

print("Delay Rate:", Delay_rate, "%")

Delay Rate: 100.0 %


#### 2.Average Delay Hours by Supplier

##### Average, how many hours are shipments delayed for each supplier?

In [10]:
print(shipment.columns.tolist())

['Shipment_ID', 'Order_ID', 'Shipment_Date', 'Delivery_Date', 'Delivery_Time', 'Supplier_ID', 'Warehouse_ID']


In [28]:
supplier_business = shipment.merge(business, on="Shipment_ID", how="left")
supplier_delay = supplier_business.groupby("Supplier_ID")["Delay_Hours"].mean()

In [29]:
supplier_delay

Supplier_ID
SUP0001    6.210274
SUP0002    5.951301
SUP0003    6.288222
SUP0004    6.156346
SUP0005    5.916738
             ...   
SUP0096    6.239917
SUP0097    5.925495
SUP0098    5.741115
SUP0099    5.968153
SUP0100    5.673921
Name: Delay_Hours, Length: 100, dtype: float64

##### Average Delay Hours by Supplier calculates the average number of hours that shipments are delayed for each supplier.
##### In my dataset, SUP0001 has an average delay of 6.21 hours, 
##### meaning its shipments are delayed by approximately 6.21 hours on average.

In [16]:
(supplier_delay.max())

np.float64(6.400872374798062)

In [17]:
(supplier_delay.min())

np.float64(5.496195286195286)

#### 3.Average Delay Hours by Route

##### Which routes have higher average shipment delays?

In [20]:
(transport.columns)

Index(['Shipment_ID', 'Transport_Mode', 'Route_Distance', 'Fuel_Cost'], dtype='str')

In [25]:
route_data["Route_Group"] = pd.cut( route_data["Route_Distance"],
    bins=[0, 500, 1000, 1500, 2000, 2500],
    labels=["0-500 km", "500-1000 km", "1000-1500 km", "1500-2000 km", "2000-2500 km"])
route_delay = route_data.groupby("Route_Group",observed=True)["Delay_Hours"].mean()

In [26]:
route_delay

Route_Group
0-500 km        6.013586
500-1000 km     6.077691
1000-1500 km    5.962338
1500-2000 km    6.052867
2000-2500 km    5.976617
Name: Delay_Hours, dtype: float64

##### Route distances were divided into five distance groups. 
##### The average delay was calculated for each group. 
##### The average delay ranges from approximately 5.96 to 6.08 hours across the route groups.

#### 4.Supplier Reliability Score

In [40]:
supplier_delay = (supplier_business.groupby("Supplier_ID")["Delay_Hours"].mean())

In [42]:
supplier_data = supplier_delay.reset_index()

supplier_data["Reliability_Score"] = (
    100 - (supplier_data["Delay_Hours"] * 10))

In [43]:
supplier_data

,Supplier_ID,Delay_Hours,Reliability_Score
0,SUP0001,6.210274,37.897258
1,SUP0002,5.951301,40.486985
2,SUP0003,6.288222,37.117778
3,SUP0004,6.156346,38.436545
4,SUP0005,5.916738,40.832620
...,...,...,...
95,SUP0096,6.239917,37.600825
96,SUP0097,5.925495,40.745050
97,SUP0098,5.741115,42.588852
98,SUP0099,5.968153,40.318475


##### Supplier reliability varies based on average delay hours. Suppliers with lower average delay hours have higher reliability scores,
##### while suppliers with higher average delays have lower reliability scores. In this dataset, most suppliers have similar average delays hours.
##### Supplier reliability is inversely related to average delay. Lower delay indicates better supplier reliability.

#### 5.Route Risk Score

In [47]:
route_risk_score = (route_delay * 10).round(2)

In [48]:
route_risk_score

Route_Group
0-500 km        60.14
500-1000 km     60.78
1000-1500 km    59.62
1500-2000 km    60.53
2000-2500 km    59.77
Name: Delay_Hours, dtype: float64

##### The Route Risk Score is based on the average delay hours for each route group.
##### Higher average delay indicates higher route risk, while lower average delay indicates lower route risk.

#### 6.Transportation Efficiency Metric

##### how much fuel cost is spent for each kilometer of transportation?
##### Formula: Transportation Efficiency = Fuel Cost ÷ Route Distance

In [51]:
transport["Transport_Efficiency"] = (
    transport["Fuel_Cost"] / transport["Route_Distance"]
).round(2)

In [52]:
transport["Transport_Efficiency"]

0        4.16
1        7.42
2        6.77
3        5.27
4        5.54
         ... 
59995    3.19
59996    6.81
59997    6.82
59998    5.81
59999    4.02
Name: Transport_Efficiency, Length: 60000, dtype: float64

In [53]:
transport["Transport_Efficiency"].head()

0    4.16
1    7.42
2    6.77
3    5.27
4    5.54
Name: Transport_Efficiency, dtype: float64

In [56]:
transport["Transport_Efficiency"].describe().round(2)

count    60000.00
mean         5.50
std          1.45
min          3.00
25%          4.24
50%          5.50
75%          6.75
max          8.00
Name: Transport_Efficiency, dtype: float64

##### The Transportation Efficiency Metric shows the fuel cost incurred per kilometer.
##### A lower value indicates lower fuel cost per kilometer, while a higher value indicates higher fuel cost per kilometer. 
##### This metric can be used to compare transportation cost efficiency across different shipment routes or transport modes.